# Action Label EDA: LLM-Generated Labels + IMU Validation

This notebook analyzes the action labels generated by Qwen3-8B and validates them against IMU sensor data.

## Contents
1. Load and inspect labeled data
2. Label distribution analysis
3. Scenario-specific patterns
4. IMU correlation analysis
5. Quality metrics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
LABELS_PATH = '../data/labels/action_labels_llm_clean.csv'
IMU_DIR = Path('../data/ego4d_data/v2/imu')
SCENARIO_LABELS_PATH = '../data/labels/scenario_labels.csv'

## 1. Load and Inspect Data

In [ ]:
# Load labels
df = pd.read_csv(LABELS_PATH)

print(f"Total labeled narrations: {len(df):,}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display sample
df.head(10)

In [ ]:
# Basic statistics
print("=== Dataset Summary ===")
print(f"Unique videos: {df['video_uid'].nunique():,}")
print(f"Unique scenarios: {df['scenario'].nunique()}")
print(f"Unique action labels: {df['action'].nunique()}")
print(f"\nTimestamp range: {df['timestamp_sec'].min():.2f}s - {df['timestamp_sec'].max():.2f}s")
print(f"Average narrations per video: {len(df) / df['video_uid'].nunique():.1f}")

## 2. Label Distribution Analysis

In [ ]:
# Overall label distribution
label_counts = df['action'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart
label_counts.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Action Label Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Action Label')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Add percentage labels
for i, v in enumerate(label_counts):
    ax1.text(i, v + 100, f'{v/len(df)*100:.1f}%', ha='center', va='bottom')

# Pie chart
colors = sns.color_palette('Set2', len(label_counts))
ax2.pie(label_counts, labels=label_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
ax2.set_title('Action Label Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Label Counts ===")
print(label_counts)
print(f"\n=== Percentages ===")
print((label_counts / len(df) * 100).round(2))

## 3. Scenario-Specific Analysis

In [ ]:
# Scenario distribution
scenario_counts = df['scenario'].value_counts()

plt.figure(figsize=(12, 6))
scenario_counts.plot(kind='barh', color='coral')
plt.title('Narrations per Scenario', fontsize=14, fontweight='bold')
plt.xlabel('Count')
plt.ylabel('Scenario')
plt.grid(axis='x', alpha=0.3)

# Add count labels
for i, v in enumerate(scenario_counts):
    plt.text(v + 50, i, f'{v:,}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Cross-tabulation: Scenario vs Action
ct = pd.crosstab(df['scenario'], df['action'], normalize='index') * 100

plt.figure(figsize=(14, 8))
sns.heatmap(ct, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Percentage (%)'})
plt.title('Action Distribution by Scenario (%)', fontsize=14, fontweight='bold')
plt.xlabel('Action Label')
plt.ylabel('Scenario')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\n=== Raw Counts ===")
print(pd.crosstab(df['scenario'], df['action']))

## 4. Label Quality Checks

In [ ]:
# Check for unknown/error labels
unknown_labels = df[df['action'].str.contains('Unknown|Error', case=False, na=False)]

print(f"Labels with 'Unknown' or 'Error': {len(unknown_labels):,} ({len(unknown_labels)/len(df)*100:.2f}%)")

if len(unknown_labels) > 0:
    print("\nSample of problematic labels:")
    print(unknown_labels[['narration_text', 'scenario', 'action', 'reasoning']].head(10))

In [ ]:
# Check reasoning length distribution
df['reasoning_length'] = df['reasoning'].str.len()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
ax1.hist(df['reasoning_length'], bins=50, color='skyblue', edgecolor='black')
ax1.set_title('Reasoning Length Distribution', fontsize=12, fontweight='bold')
ax1.set_xlabel('Characters')
ax1.set_ylabel('Frequency')
ax1.axvline(df['reasoning_length'].median(), color='red', linestyle='--', label=f'Median: {df["reasoning_length"].median():.0f}')
ax1.legend()

# Box plot by action
df.boxplot(column='reasoning_length', by='action', ax=ax2)
ax2.set_title('Reasoning Length by Action Label')
ax2.set_xlabel('Action')
ax2.set_ylabel('Reasoning Length (chars)')
ax2.tick_params(axis='x', rotation=45)
plt.suptitle('')  # Remove auto-title

plt.tight_layout()
plt.show()

print(f"\nReasoning length stats:\n{df['reasoning_length'].describe()}")

## 5. IMU Data Correlation (Sample Analysis)

We'll load IMU data for a few videos and check if the labels correlate with expected motion patterns.

In [ ]:
# Find available IMU files
imu_files = list(IMU_DIR.glob('*.csv'))
print(f"Found {len(imu_files)} IMU files locally")

if len(imu_files) > 0:
    print("\nSample IMU files:")
    for f in imu_files[:5]:
        print(f"  {f.name}")
else:
    print("\nNo IMU files found. Skipping correlation analysis.")

In [ ]:
# Function to load and preprocess IMU data
def load_imu_data(video_uid):
    """Load IMU data for a specific video"""
    imu_file = IMU_DIR / f"{video_uid}.csv"
    if not imu_file.exists():
        return None
    
    imu = pd.read_csv(imu_file)
    
    # Calculate magnitude of acceleration
    if all(col in imu.columns for col in ['accel_x', 'accel_y', 'accel_z']):
        imu['accel_mag'] = np.sqrt(imu['accel_x']**2 + imu['accel_y']**2 + imu['accel_z']**2)
    
    # Calculate magnitude of gyroscope
    if all(col in imu.columns for col in ['gyro_x', 'gyro_y', 'gyro_z']):
        imu['gyro_mag'] = np.sqrt(imu['gyro_x']**2 + imu['gyro_y']**2 + imu['gyro_z']**2)
    
    return imu

# Select a video with IMU data
if len(imu_files) > 0:
    sample_uid = imu_files[0].stem
    print(f"Analyzing video: {sample_uid}")
    
    # Load IMU
    imu_data = load_imu_data(sample_uid)
    
    if imu_data is not None:
        print(f"\nIMU data shape: {imu_data.shape}")
        print(f"Columns: {imu_data.columns.tolist()}")
        print(f"\nSample:")
        print(imu_data.head())
    else:
        print("Failed to load IMU data")

In [ ]:
# Visualize IMU + Labels for sample video
if len(imu_files) > 0 and imu_data is not None:
    # Get labels for this video
    video_labels = df[df['video_uid'] == sample_uid].sort_values('timestamp_sec')
    
    print(f"Found {len(video_labels)} labels for this video")
    
    if len(video_labels) > 0 and 'timestamp' in imu_data.columns:
        fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
        
        # Plot acceleration magnitude
        if 'accel_mag' in imu_data.columns:
            axes[0].plot(imu_data['timestamp'], imu_data['accel_mag'], 
                        linewidth=0.5, alpha=0.7, color='blue')
            axes[0].set_ylabel('Accel Magnitude (m/s²)')
            axes[0].set_title(f'IMU Data + Action Labels: {sample_uid}', fontweight='bold')
            axes[0].grid(alpha=0.3)
        
        # Plot gyro magnitude
        if 'gyro_mag' in imu_data.columns:
            axes[1].plot(imu_data['timestamp'], imu_data['gyro_mag'], 
                        linewidth=0.5, alpha=0.7, color='green')
            axes[1].set_ylabel('Gyro Magnitude (rad/s)')
            axes[1].grid(alpha=0.3)
        
        # Plot labels as colored regions
        label_colors = {
            'Locomotion': 'red',
            'Essential Operation': 'orange',
            'Object Transfer': 'yellow',
            'Search': 'green',
            'Error / Correction': 'purple',
            'Stationary': 'gray'
        }
        
        for idx, row in video_labels.iterrows():
            color = label_colors.get(row['action'], 'black')
            for ax in axes[:2]:
                ax.axvline(row['timestamp_sec'], color=color, alpha=0.3, linewidth=2)
        
        # Create legend
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=color, label=label) 
                          for label, color in label_colors.items() 
                          if label in video_labels['action'].values]
        axes[0].legend(handles=legend_elements, loc='upper right', fontsize=8)
        
        # Timeline of labels
        axes[2].scatter(video_labels['timestamp_sec'], 
                       [label_colors.get(a, 'black') for a in video_labels['action']], 
                       c=[label_colors.get(a, 'black') for a in video_labels['action']], 
                       s=100, alpha=0.6)
        axes[2].set_ylabel('Action Labels')
        axes[2].set_xlabel('Time (seconds)')
        axes[2].set_yticks([])
        axes[2].grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print sample labels
        print("\nSample labels for this video:")
        print(video_labels[['timestamp_sec', 'narration_text', 'action']].head(10))

## 6. Statistical Validation: Label vs IMU Patterns

Calculate average motion magnitude for different action labels.

In [ ]:
# For videos with IMU data, calculate motion metrics around labeled timestamps
motion_by_label = []

# Sample 10 videos with IMU data
sample_videos = [f.stem for f in imu_files[:10]]

for video_uid in sample_videos:
    imu = load_imu_data(video_uid)
    if imu is None or 'accel_mag' not in imu.columns:
        continue
    
    video_labels = df[df['video_uid'] == video_uid]
    
    for _, label_row in video_labels.iterrows():
        ts = label_row['timestamp_sec']
        
        # Get IMU window around this timestamp (±1 second)
        if 'timestamp' in imu.columns:
            window = imu[(imu['timestamp'] >= ts - 1) & (imu['timestamp'] <= ts + 1)]
            
            if len(window) > 0:
                motion_by_label.append({
                    'action': label_row['action'],
                    'scenario': label_row['scenario'],
                    'accel_mean': window['accel_mag'].mean(),
                    'accel_std': window['accel_mag'].std(),
                    'gyro_mean': window.get('gyro_mag', pd.Series([0])).mean() if 'gyro_mag' in window.columns else 0
                })

motion_df = pd.DataFrame(motion_by_label)

if len(motion_df) > 0:
    print(f"Collected motion data for {len(motion_df)} labeled moments")
    
    # Group by action and calculate statistics
    motion_stats = motion_df.groupby('action').agg({
        'accel_mean': ['mean', 'std', 'count'],
        'gyro_mean': ['mean', 'std']
    }).round(3)
    
    print("\n=== Motion Statistics by Action Label ===")
    print(motion_stats)
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    motion_df.boxplot(column='accel_mean', by='action', ax=ax1)
    ax1.set_title('Acceleration by Action Label')
    ax1.set_ylabel('Mean Acceleration (m/s²)')
    ax1.tick_params(axis='x', rotation=45)
    plt.suptitle('')
    
    motion_df.boxplot(column='gyro_mean', by='action', ax=ax2)
    ax2.set_title('Gyroscope by Action Label')
    ax2.set_ylabel('Mean Gyroscope (rad/s)')
    ax2.tick_params(axis='x', rotation=45)
    plt.suptitle('')
    
    plt.tight_layout()
    plt.show()
else:
    print("No motion data collected. Check if IMU files have 'timestamp' column.")

## 7. Summary and Insights

In [ ]:
print("=" * 60)
print("SUMMARY: Action Label EDA")
print("=" * 60)

print(f"\n📊 Dataset Statistics:")
print(f"  - Total narrations: {len(df):,}")
print(f"  - Unique videos: {df['video_uid'].nunique():,}")
print(f"  - Scenarios: {df['scenario'].nunique()}")
print(f"  - Action labels: {df['action'].nunique()}")

print(f"\n🏷️  Top 3 Action Labels:")
for i, (label, count) in enumerate(label_counts.head(3).items(), 1):
    print(f"  {i}. {label}: {count:,} ({count/len(df)*100:.1f}%)")

print(f"\n🎯 Top 3 Scenarios:")
for i, (scenario, count) in enumerate(scenario_counts.head(3).items(), 1):
    print(f"  {i}. {scenario}: {count:,} narrations")

if len(motion_df) > 0:
    print(f"\n📈 IMU Validation:")
    print(f"  - Analyzed {len(sample_videos)} videos with IMU data")
    print(f"  - Collected {len(motion_df)} motion samples")
    
    # Expected patterns
    high_motion = motion_df.groupby('action')['accel_mean'].mean().nlargest(3)
    print(f"\n  Highest motion labels:")
    for label, accel in high_motion.items():
        print(f"    - {label}: {accel:.2f} m/s²")

print(f"\n✅ Quality Metrics:")
print(f"  - Unknown/Error labels: {len(unknown_labels):,} ({len(unknown_labels)/len(df)*100:.2f}%)")
print(f"  - Avg reasoning length: {df['reasoning_length'].mean():.0f} chars")

print("\n" + "=" * 60)